# DCA Ingest Gantenbein (Separated TTL Outputs)

This notebook generates **separate TTL files** for:
- DROID base metadata
- EXIF/XMP enrichment delta
- PROV events and agents

`enriched.ttl` is intentionally not generated in this workflow.

In [2]:
from pathlib import Path
from urllib.parse import quote
import json
import subprocess
import hashlib
import shutil
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, RDFS, XSD, DCTERMS

In [3]:
# Paths
BASE = Path('/home/renku/work/dcaonnextcloud-500gb')
SOURCE_DIR = BASE / 'DigitalMaterialCopies/WeingutGantenbein/gramazio-kohler-archiv-server'
RESULT_DIR = BASE / 'dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

NEXTCLOUD_DAV_BASE = 'https://nextcloud.ethz.ch/remote.php/dav/files/padrian/DCA'

DROID_CSV = RESULT_DIR / 'gramazio-kohler-archiv-server_DROIDresults.csv'
OUT_DROID = RESULT_DIR / '036_WeingutGantenbein_DROID.ttl'
OUT_EXIFXMP = RESULT_DIR / '036_WeingutGantenbein_EXIFXMP.ttl'
OUT_PROV = RESULT_DIR / '036_WeingutGantenbein_prov.ttl'

EXIFTOOL = 'exiftool'
XMP_JSON = RESULT_DIR / 'gramazio-kohler-archiv-server_xmp.json'

print('SOURCE_DIR:', SOURCE_DIR)
print('DROID_CSV :', DROID_CSV)
print('OUT_DROID :', OUT_DROID)
print('OUT_EXIFXMP:', OUT_EXIFXMP)
print('OUT_PROV  :', OUT_PROV)
print('NEXTCLOUD_DAV_BASE:', NEXTCLOUD_DAV_BASE)
print('EXIFTOOL:', EXIFTOOL)
print('SYSTEM_EXIFTOOL:', shutil.which(EXIFTOOL))

SOURCE_DIR: /home/renku/work/dcaonnextcloud-500gb/DigitalMaterialCopies/WeingutGantenbein/gramazio-kohler-archiv-server
DROID_CSV : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/gramazio-kohler-archiv-server_DROIDresults.csv
OUT_DROID : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_DROID.ttl
OUT_EXIFXMP: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_EXIFXMP.ttl
OUT_PROV  : /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_prov.ttl
NEXTCLOUD_DAV_BASE: https://nextcloud.ethz.ch/remote.php/dav/files/padrian/DCA
EXIFTOOL: exiftool
SYSTEM_EXIFTOOL: /layers/paketo-buildpacks_conda-env-update/conda-env/bin/exiftool


In [4]:
# Namespaces
DCA = Namespace('http://dca.ethz.ch/ontology#')
DCA_ID = Namespace('http://dca.ethz.ch/id/')
PREMIS = Namespace('http://www.loc.gov/premis/rdf/v3/')
RICO = Namespace('https://www.ica.org/standards/RiC/ontology#')

def make_graph():
    g = Graph()
    g.bind('dca', DCA)
    g.bind('dca-id', DCA_ID)
    g.bind('premis', PREMIS)
    g.bind('rico', RICO)
    g.bind('dcterms', DCTERMS)
    g.bind('xsd', XSD)
    return g

def file_uri_from_md5(md5_value: str, fallback_path: str) -> URIRef:
    if isinstance(md5_value, str) and md5_value.strip():
        return URIRef(f'http://dca.ethz.ch/id/file_{md5_value.strip()[:16]}')
    digest = hashlib.sha1(str(fallback_path).encode('utf-8')).hexdigest()
    return URIRef(f'http://dca.ethz.ch/id/file_{digest[:16]}')

def candidate_lookup_keys(path_value: str):
    if not isinstance(path_value, str) or not path_value.strip():
        return []
    value = path_value.strip().replace('\\', '/')
    keys = {value, value.replace('!/', '/')}
    if value.startswith('zip:'):
        bare = value[4:]
        keys.add(bare)
        keys.add(bare.replace('!/', '/'))
    return [key for key in keys if key]

def local_path_to_nextcloud_uri(path_value: str):
    if not isinstance(path_value, str) or not path_value.strip():
        return None
    try:
        rel = Path(path_value).relative_to(BASE)
    except Exception:
        return None
    encoded_rel = '/'.join(quote(part, safe='') for part in rel.parts)
    return URIRef(f"{NEXTCLOUD_DAV_BASE}/{encoded_rel}")

def add_identifier(g: Graph, file_uri: URIRef, id_type: str, id_value: str):
    if not id_value:
        return
    b = BNode()
    g.add((file_uri, PREMIS.hasIdentifier, b))
    g.add((b, PREMIS.identifierType, Literal(id_type)))
    g.add((b, PREMIS.identifierValue, Literal(str(id_value))))

## 1) DROID CSV -> DROID TTL (base graph only)

In [ ]:
droid_df = pd.read_csv(DROID_CSV, low_memory=False)
g_droid = make_graph()
project_uri = URIRef('http://dca.ethz.ch/id/project_WeingutGantenbein')

for _, row in droid_df.iterrows():
    file_path = str(row.get('FILE_PATH', ''))
    file_name = str(row.get('NAME', '') or Path(file_path).name)
    md5_hash = str(row.get('HASH', '') or '')

    file_uri = file_uri_from_md5(md5_hash, file_path)
    g_droid.add((file_uri, RDF.type, DCA.ArchiveFile))
    g_droid.add((file_uri, RDF.type, PREMIS.Object))
    g_droid.add((file_uri, RDF.type, RICO.Record))
    g_droid.add((file_uri, DCA.belongsToProject, project_uri))

    if file_path:
        nextcloud_uri = local_path_to_nextcloud_uri(file_path)
        if nextcloud_uri is not None:
            g_droid.add((file_uri, DCTERMS.identifier, nextcloud_uri))
        else:
            g_droid.add((file_uri, DCTERMS.identifier, Literal(file_path)))
    if file_name:
        g_droid.add((file_uri, DCTERMS.title, Literal(file_name)))

    modified = row.get('LAST_MODIFIED')
    if isinstance(modified, str) and modified:
        g_droid.add((file_uri, DCTERMS.modified, Literal(modified, datatype=XSD.dateTime)))

    fmt_name = row.get('FORMAT_NAME')
    if isinstance(fmt_name, str) and fmt_name:
        g_droid.add((file_uri, PREMIS.hasFormatName, Literal(fmt_name)))

    mime = row.get('MIME_TYPE')
    puid = row.get('PUID')
    if isinstance(mime, str) and mime:
        g_droid.add((file_uri, PREMIS.hasFormatNote, Literal(f'MIME: {mime}')))
    if isinstance(puid, str) and puid:
        g_droid.add((file_uri, PREMIS.hasFormatNote, Literal(f'PRONOM ID: {puid}')))

    size = row.get('SIZE')
    if pd.notna(size):
        g_droid.add((file_uri, PREMIS.hasSize, Literal(int(size), datatype=XSD.long)))

    g_droid.add((file_uri, PREMIS.hasCreatingApplication, Literal('DROID: Signature')))
    add_identifier(g_droid, file_uri, 'MD5 Hash', md5_hash if md5_hash else '')

g_droid.serialize(OUT_DROID, format='turtle')
print('DROID triples:', len(g_droid))
print('Saved:', OUT_DROID)

## 2) EXIF/XMP -> EXIFXMP TTL (delta only)

In [10]:
if shutil.which(EXIFTOOL) is None:
    raise RuntimeError(
        "ExifTool ist nicht installiert. Bitte einmalig in Renku ausführen:\n"
        "micromamba install -y -c conda-forge exiftool"
    )

if 'droid_df' not in globals():
    droid_df = pd.read_csv(DROID_CSV, low_memory=False)

# Quick-check mode: set an integer (e.g. 100/300) for a partial run, or None for full run.
DEBUG_SAMPLE_SIZE = None

# Request relevant metadata tags explicitly so sample/full runs behave consistently.
# Using explicit tag requests avoids environment-dependent default tag sets.
EXIF_BASE_ARGS = [
    EXIFTOOL, '-json', '-a', '-u', '-m', '-G1', '-s',
    '-XMP:CreatorTool',
    '-XMP-xmp:CreatorTool',
    '-XMP-xmpMM:DocumentID',
    '-XMP-xmpMM:InstanceID',
    '-XMP:DocumentID',
    '-XMP:InstanceID',
    '-EXIF:Software',
    '-PNG:Software',
    '-PDF:Creator',
    '-PDF:Producer',
    '-QuickTime:SoftwareVersion',
    '-File:FileType',
    '-File:MIMEType',
]

droid_hash_by_path = {}
for _, row in droid_df.iterrows():
    file_path = str(row.get('FILE_PATH', '') or '')
    md5_hash = str(row.get('HASH', '') or '').strip()
    if not file_path or not md5_hash:
        continue
    for key in candidate_lookup_keys(file_path):
        droid_hash_by_path.setdefault(key, md5_hash)

if DEBUG_SAMPLE_SIZE is None:
    cmd = [*EXIF_BASE_ARGS, '-r', '-recurse_archives', str(SOURCE_DIR)]
    scan_mode = 'full'
    sampled_paths = []
else:
    # Use only real files (no directories, no zip: virtual paths) for a fast sanity check.
    mask = (
        droid_df['FILE_PATH'].astype(str).str.startswith(str(SOURCE_DIR))
        & ~droid_df['FILE_PATH'].astype(str).str.startswith('zip:')
    )
    sampled_paths = []
    for path_value in droid_df.loc[mask, 'FILE_PATH'].head(DEBUG_SAMPLE_SIZE * 5):
        path_str = str(path_value)
        p = Path(path_str)
        if p.exists() and p.is_file():
            sampled_paths.append(path_str)
        if len(sampled_paths) >= DEBUG_SAMPLE_SIZE:
            break

    if not sampled_paths:
        raise RuntimeError(
            'Quick-check mode found no existing files. '
            'Set DEBUG_SAMPLE_SIZE=None for full recursive scan.'
        )

    cmd = [*EXIF_BASE_ARGS, *sampled_paths]
    scan_mode = 'sample'

result = subprocess.run(cmd, capture_output=True, text=True, check=False)
if result.returncode != 0 and not result.stdout.strip():
    stderr_tail = '\n'.join(result.stderr.splitlines()[-30:])
    raise RuntimeError(f'ExifTool failed (rc={result.returncode}) with no JSON output:\n{stderr_tail}')

XMP_JSON.write_text(result.stdout, encoding='utf-8')
xmp_rows = json.loads(result.stdout)

def value_to_text(v):
    if isinstance(v, str):
        return v.strip()
    if isinstance(v, (int, float, bool)):
        return str(v).strip()
    if isinstance(v, list):
        parts = [value_to_text(item) for item in v]
        parts = [p for p in parts if p]
        return '; '.join(parts).strip()
    return ''

def normalize_key(key):
    key = str(key)
    for ch in (':', '-', ' ', '_', '/', '[', ']', '(', ')', '.'):
        key = key.replace(ch, '')
    return key.lower()

def pick_value(rec, explicit_keys=(), token_groups=()):
    # 1) Prefer exact keys when present.
    for k in explicit_keys:
        if k in rec:
            text = value_to_text(rec.get(k))
            if text:
                return text, k

    # 2) Fallback: find keys whose normalized name contains all required tokens.
    for rk, rv in rec.items():
        text = value_to_text(rv)
        if not text:
            continue
        nk = normalize_key(rk)
        for tokens in token_groups:
            if all(token in nk for token in tokens):
                return text, rk

    return None, None

g_xmp = make_graph()
matched_via_droid = 0
matched_via_exif = 0
fallback_via_path = 0
rows_with_any_meta = 0
rows_with_app = 0
rows_with_docid = 0
rows_with_instid = 0
key_hits = {'app': {}, 'docid': {}, 'instid': {}}
missing_meta_key_samples = []

for rec in xmp_rows:
    src = str(rec.get('SourceFile', ''))

    md5 = ''
    for key in candidate_lookup_keys(src):
        md5 = droid_hash_by_path.get(key, '')
        if md5:
            matched_via_droid += 1
            break

    if not md5:
        md5 = str(rec.get('MD5Digest', '') or rec.get('MD5', '') or '')
        if md5:
            matched_via_exif += 1
        else:
            fallback_via_path += 1

    file_uri = file_uri_from_md5(md5, src)
    g_xmp.add((file_uri, RDF.type, DCA.ArchiveFile))

    app, app_key = pick_value(
        rec,
        explicit_keys=(
            'XMP:CreatorTool', 'XMP-xmp:CreatorTool', 'CreatorTool',
            'EXIF:Software', 'PNG:Software', 'PDF:Creator', 'PDF:Producer',
            'QuickTime:SoftwareVersion', 'Software', 'Application'
        ),
        token_groups=(('creatortool',), ('software',), ('application',), ('producer',), ('creator',))
    )
    docid, docid_key = pick_value(
        rec,
        explicit_keys=('XMP-xmpMM:DocumentID', 'XMP:DocumentID', 'DocumentID'),
        token_groups=(('documentid',),)
    )
    instid, instid_key = pick_value(
        rec,
        explicit_keys=('XMP-xmpMM:InstanceID', 'XMP:InstanceID', 'InstanceID'),
        token_groups=(('instanceid',),)
    )

    has_meta = False
    if app:
        g_xmp.add((file_uri, PREMIS.hasCreatingApplication, Literal(app)))
        rows_with_app += 1
        key_hits['app'][app_key or 'unknown'] = key_hits['app'].get(app_key or 'unknown', 0) + 1
        has_meta = True

    if docid:
        add_identifier(g_xmp, file_uri, 'XMP DocumentID', docid)
        rows_with_docid += 1
        key_hits['docid'][docid_key or 'unknown'] = key_hits['docid'].get(docid_key or 'unknown', 0) + 1
        has_meta = True

    if instid:
        add_identifier(g_xmp, file_uri, 'XMP InstanceID', instid)
        rows_with_instid += 1
        key_hits['instid'][instid_key or 'unknown'] = key_hits['instid'].get(instid_key or 'unknown', 0) + 1
        has_meta = True

    if has_meta:
        rows_with_any_meta += 1
    elif len(missing_meta_key_samples) < 10:
        missing_meta_key_samples.append({
            'SourceFile': src,
            'available_keys_sample': sorted(list(rec.keys()))[:60]
        })

diag_path = RESULT_DIR / 'exif_key_diagnostics.json'
diag_path.write_text(json.dumps({
    'scan_mode': scan_mode,
    'debug_sample_size': DEBUG_SAMPLE_SIZE,
    'sampled_paths_count': len(sampled_paths),
    'exiftool_return_code': result.returncode,
    'exiftool_stderr_tail': result.stderr.splitlines()[-20:],
    'total_rows': len(xmp_rows),
    'rows_with_any_meta': rows_with_any_meta,
    'rows_with_app': rows_with_app,
    'rows_with_docid': rows_with_docid,
    'rows_with_instid': rows_with_instid,
    'key_hits': key_hits,
    'id_source_stats': {
        'matched_via_droid': matched_via_droid,
        'matched_via_exif': matched_via_exif,
        'fallback_via_path': fallback_via_path,
    },
    'missing_meta_key_samples': missing_meta_key_samples,
}, indent=2), encoding='utf-8')

g_xmp.serialize(OUT_EXIFXMP, format='turtle')
print('EXIF/XMP triples:', len(g_xmp))
print('Saved:', OUT_EXIFXMP)
print('Diagnostics:', diag_path)
print('Meta stats:', {
    'scan_mode': scan_mode,
    'debug_sample_size': DEBUG_SAMPLE_SIZE,
    'sampled_paths_count': len(sampled_paths),
    'exiftool_return_code': result.returncode,
    'total_rows': len(xmp_rows),
    'rows_with_any_meta': rows_with_any_meta,
    'rows_with_app': rows_with_app,
    'rows_with_docid': rows_with_docid,
    'rows_with_instid': rows_with_instid,
})
print('ID source stats:', {
    'matched_via_droid': matched_via_droid,
    'matched_via_exif': matched_via_exif,
    'fallback_via_path': fallback_via_path,
})
print('Top key hits:', {
    'app': sorted(key_hits['app'].items(), key=lambda x: x[1], reverse=True)[:5],
    'docid': sorted(key_hits['docid'].items(), key=lambda x: x[1], reverse=True)[:5],
    'instid': sorted(key_hits['instid'].items(), key=lambda x: x[1], reverse=True)[:5],
})

EXIF/XMP triples: 38743
Saved: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/036_WeingutGantenbein_EXIFXMP.ttl
Diagnostics: /home/renku/work/dcaonnextcloud-500gb/dca-metadataraw/WeingutGantenbein/gramazio-kohler-archiv-server_results/exif_key_diagnostics.json
Meta stats: {'scan_mode': 'full', 'debug_sample_size': None, 'sampled_paths_count': 0, 'exiftool_return_code': 0, 'total_rows': 12485, 'rows_with_any_meta': 7928, 'rows_with_app': 7928, 'rows_with_docid': 6119, 'rows_with_instid': 85}
ID source stats: {'matched_via_droid': 12485, 'matched_via_exif': 0, 'fallback_via_path': 0}
Top key hits: {'app': [('XMP-xmp:CreatorTool', 6118), ('IFD0:Software', 1781), ('PDF:Creator', 26), ('PDF:Producer', 2), ('UserData:SoftwareVersion', 1)], 'docid': [('XMP-xmpMM:DocumentID', 6119)], 'instid': [('XMP-xmpMM:InstanceID', 85)]}


## 3) Provenance TTL (events and agents only)

In [ ]:
g_prov = make_graph()
activity_droid = URIRef('http://dca.ethz.ch/id/activity_DROID_Conversion_2026')
activity_xmp = URIRef('http://dca.ethz.ch/id/activity_EXIFXMP_Integration_2026')
agent_droid = URIRef('http://dca.ethz.ch/id/agent_DROID')
agent_exiftool = URIRef('http://dca.ethz.ch/id/agent_ExifTool')

g_prov.add((activity_droid, RDF.type, PREMIS.Event))
g_prov.add((activity_droid, RDFS.label, Literal('DROID to RDF Conversion')))
g_prov.add((activity_droid, DCTERMS.date, Literal('2026-03-31', datatype=XSD.date)))

g_prov.add((activity_xmp, RDF.type, PREMIS.Event))
g_prov.add((activity_xmp, RDFS.label, Literal('EXIF/XMP Integration')))
g_prov.add((activity_xmp, DCTERMS.date, Literal('2026-03-31', datatype=XSD.date)))

g_prov.add((agent_droid, RDF.type, PREMIS.Agent))
g_prov.add((agent_droid, RDFS.label, Literal('DROID')))

g_prov.add((agent_exiftool, RDF.type, PREMIS.Agent))
g_prov.add((agent_exiftool, RDFS.label, Literal('ExifTool')))

g_prov.add((activity_droid, PREMIS.hasAgent, agent_droid))
g_prov.add((activity_xmp, PREMIS.hasAgent, agent_exiftool))

g_prov.serialize(OUT_PROV, format='turtle')
print('PROV triples:', len(g_prov))
print('Saved:', OUT_PROV)

## 4) Quick validation

In [ ]:
print('Exists DROID  :', OUT_DROID.exists(), OUT_DROID)
print('Exists EXIFXMP:', OUT_EXIFXMP.exists(), OUT_EXIFXMP)
print('Exists PROV   :', OUT_PROV.exists(), OUT_PROV)

# Guardrails: PROV should not carry DROID signature payload
prov_text = OUT_PROV.read_text(encoding='utf-8') if OUT_PROV.exists() else ''
print('PROV contains DROID signature?', 'DROID: Signature' in prov_text)

# Guardrails: DROID should not carry XMP DocumentID payload
droid_text = OUT_DROID.read_text(encoding='utf-8') if OUT_DROID.exists() else ''
print('DROID contains XMP DocumentID?', 'XMP DocumentID' in droid_text)